In [ ]:
import logging
from datetime import datetime

import pandas as pd
import torch
from datasets import load_dataset

from sentence_transformers import SentenceTransformer, evaluation, losses, LoggingHandler
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

In [ ]:
# Set the log level to INFO to get more information
logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO, handlers=[LoggingHandler()])
output_path = "distilled/model-distillation-" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [ ]:
# Teacher Model: Model we want to distill to a smaller model
# teacher_model_name = "stsb-bert-base"
teacher_model_name = "sentence-transformers/stsb-roberta-base-v2"
teacher_model = SentenceTransformer(teacher_model_name)

In [ ]:
for i, layer_module in enumerate(teacher_model._first_module().auto_model.encoder.layer):
    print(i, layer_module)

In [ ]:
# Create a smaller student model by using only some of the teacher layers
student_model = SentenceTransformer(teacher_model_name)

# Get the transformer model
auto_model = student_model._first_module().auto_model

# Which layers to keep from the teacher model. We equally spread the layers to keep over the original teacher
# layers_to_keep = [0, 5, 7, 9, 10, 11]
layers_to_keep = [0, 1, 7, 9, 10, 11]
# layers_to_keep = [0, 4, 10, 11]
# layers_to_keep = [0, 7, 10, 11]


logging.info(f"Remove layers from student. Only keep these layers: {layers_to_keep}")

new_layers = torch.nn.ModuleList(
    [layer_module for i, layer_module in enumerate(auto_model.encoder.layer) if i in layers_to_keep]
)
auto_model.encoder.layer = new_layers
auto_model.config.num_hidden_layers = len(layers_to_keep)

print(f"Number of parameters in the Teacher model: {sum(p.numel() for p in teacher_model.parameters() if p.requires_grad)}")
print(f"Number of parameters in the Student model: {sum(p.numel() for p in student_model.parameters() if p.requires_grad)}")

In [ ]:
for i, layer_module in enumerate(auto_model.encoder.layer):
    print(i, layer_module)

In [ ]:
train_batch_size = 32

logging.info("Load the STSB dataset")
# Load the STSB dataset: https://huggingface.co/datasets/sentence-transformers/stsb
stsb_eval_dataset = load_dataset("sentence-transformers/stsb", split="validation")
stsb_test_dataset = load_dataset("sentence-transformers/stsb", split="test")
logging.info(stsb_eval_dataset)

In [ ]:
from datasets import load_from_disk
train_dataset = load_from_disk("datasets/distillation_train_dataset_stsb-roberta (0, 4507105)/")
eval_dataset = load_from_disk("datasets/distillation_evaluation_dataset_stsb-roberta/")

In [ ]:
len(train_dataset)

In [ ]:
len(eval_dataset)

In [ ]:
# Prepare the training loss
train_loss = losses.MSELoss(model=student_model)

# Create an STSB evaluator
dev_evaluator_stsb = EmbeddingSimilarityEvaluator(
    sentences1=stsb_eval_dataset["sentence1"],
    sentences2=stsb_eval_dataset["sentence2"],
    scores=stsb_eval_dataset["score"],
    # main_similarity=SimilarityFunction.COSINE,
    similarity_fn_names=['cosine',"dot","euclidean","manhattan"],
    name="sts-dev",
)

logging.info("Running STSB evaluation on the teacher model")
dev_evaluator_stsb(teacher_model)

In [ ]:
# We create an evaluator, that measure the Mean Squared Error (MSE) between the teacher and the student embeddings
eval_sentences = eval_dataset["sentence"]
dev_evaluator_mse = evaluation.MSEEvaluator(eval_sentences, eval_sentences, teacher_model=teacher_model)
dev_evaluator = evaluation.SequentialEvaluator([dev_evaluator_stsb, dev_evaluator_mse])

# Run the evaluator before training to get a baseline performance of the student model
dev_evaluator(student_model)

In [ ]:
# Define the training arguments
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir=output_path,
    
    # Optional training parameters:
    num_train_epochs=3,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=train_batch_size,
    warmup_ratio=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    # metric_for_best_model="eval_sts-dev_spearman_dot",
    # load_best_model_at_end=True,
    learning_rate=1e-5,
    
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=5000,
    save_strategy="steps",
    save_steps=5000,
    # save_total_limit=2,
    # logging_steps=1000
)

In [ ]:
# Create the trainer & start training
trainer = SentenceTransformerTrainer(
    model=student_model,
    args=args,
    train_dataset=train_dataset.select(range(0, 1000000)), # Select range of data from dataset
    eval_dataset=eval_dataset,
    loss=train_loss,
    evaluator=dev_evaluator,
)
trainer.train()

In [ ]:
dev_evaluator(student_model)
# Save the trained & evaluated model locally
final_output_dir = f"{output_path}/final"
student_model.save(final_output_dir)